<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>01. 🌐 Web Scraper with BeautifulSoup</b>
</div>


### 📌 Project Overview
In this project, we build a structured, production-ready web scraper using `requests` and `BeautifulSoup`.
The script targets the sandboxed website [Quotes to Scrape](http://quotes.toscrape.com), retrieving quote texts, their authors, and tags, paginating through multiple pages, handling connections gracefully, and writing the final dataset into a clean CSV file.

#### 🔑 Key Concepts Covered:
- Performing HTTP GET requests with the `requests` library
- Inspecting HTML page elements using CSS selectors & classes in `BeautifulSoup`
- Pagination loops and rate-limiting
- Persisting tabular data using the standard `csv` module
- Clean HTTP response code handling and connection timeout boundaries


In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time

def scrape_all_quotes():
    base_url = 'http://quotes.toscrape.com'
    page_num = 1
    all_quotes = []
    
    print('🚀 Initiating quotes scraper...')
    while True:
        url = f'{base_url}/page/{page_num}/'
        print(f'Fetching page {page_num}: {url}')
        
        try:
            # Set timeout to prevent indefinite hanging
            response = requests.get(url, timeout=10)
            if response.status_code != 200:
                print(f'🛑 Stopping. Page {page_num} returned code {response.status_code}')
                break
        except requests.RequestException as e:
            print(f'❌ Network exception encountered: {e}')
            break
            
        soup = BeautifulSoup(response.text, 'html.parser')
        quote_elements = soup.find_all('div', class_='quote')
        
        # Break loop if no quotes are found
        if not quote_elements:
            print('✨ No more quotes detected. Finalizing results.')
            break
            
        for quote_el in quote_elements:
            text = quote_el.find('span', class_='text').get_text(strip=True)
            author = quote_el.find('small', class_='author').get_text(strip=True)
            tags = [tag.get_text(strip=True) for tag in quote_el.find_all('a', class_='tag')]
            
            all_quotes.append({
                'Quote': text,
                'Author': author,
                'Tags': ', '.join(tags)
            })
            
        page_num += 1
        time.sleep(0.5)  # Rate limiting behavior to respect target server
        
    # Save database to CSV
    output_csv = 'scraped_quotes.csv'
    try:
        with open(output_csv, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=['Quote', 'Author', 'Tags'])
            writer.writeheader()
            writer.writerows(all_quotes)
        print(f'\n✅ Successfully saved {len(all_quotes)} quotes to "{output_csv}"!')
    except IOError as e:
        print(f'❌ File I/O Error: {e}')

# Execute the scraper
scrape_all_quotes()


In [ ]:
import pandas as pd
# Verify the scraping result by reading the CSV file
try:
    df = pd.read_csv('scraped_quotes.csv')
    print(f'Total quotes parsed: {len(df)}')
    print('\nFirst 5 rows of output:')
    print(df.head())
except FileNotFoundError:
    print('❌ "scraped_quotes.csv" not found. Run the scraper cell above first.')
